# **Temperature Annealing/Heating MD Directory Analysis Notebook**

This notebook provides a structured workflow for analyzing **Temperature Loop MD** simulations performed at different temperatures for a fixed number of particles and initial conditions, changing temperatures in a gradual process using thermostats. The analysis is carried out using modular Python scripts by reading the output and plotting variables which are produced by the [C++ NParticleMD program](directory/...).

---

## Workflow Overview
1. **Load and Extract Data**  
   Retrieve simulation output files based on the directory location.

2. **Visualize System Properties Over Time Steps/Temperatures**
   Plotting different variables to analyze system behavior during annealing and heating processes, including:
   - Energies (positional and rotational kinetic, potential, total)
   - Measured Temperature (positional and rotational degrees of freedom)
   - Average number of neighbors of the system for different shells and distances
   - Rotational Order Parameter ($\cos(2\Delta\Phi)$)
   - Linear and rotational velocity of the system's center of mass

3. **Visualize System Configuration**  
   Generate static and dynamic visualizations of particle configurations, allowing for direct comparison throughout the process by selecting target temperatures.

---

## Part 1: Import modules and choose directory
* Read key variables from the selected simulation directory.


In [ ]:
import numpy as np
import data_extraction
import visualization

interaction = 'OP'
output_dir = f"/home/hadis/chisurfmd/runs/heating_OP/outputs"

In [ ]:
variables = data_extraction.read_variables_batch(output_dir, while_running=True, variable_names =
    [
    'positions',
    'kinetic energy', 
    'real temperature',
    'potential energy',
    'orientation order',
    'number of neighbors',
    'center of mass angular velocity',
    'center of mass velocity',
    'handedness',
    'alignment'
])

positions = variables['positions']
kinetic_energy = variables['kinetic energy']
temperature = variables['real temperature']
potential_energy = variables['potential energy']
orientation_order = variables['orientation order']
number_of_neighbors = variables['number of neighbors']
com_ang_velocity = variables['center of mass angular velocity']
com_velocity = variables['center of mass velocity']
handedness = variables['handedness']
alignment = variables['alignment']

---

## Part 2: Visualize System Properties Over Time Steps/Temperatures
* Plot desired parameters to analyze system's trend over annealing and heating processes:

In [ ]:
visualization.plot_energies(kinetic_energy, potential_energy, temperature, show_potential=True, 
                            temperature_label=True
                            )
visualization.plot_neighbors(number_of_neighbors, temperature, temperature_label=True)
visualization.plot_order_parameter(orientation_order, temperature, temperature_label=True)
visualization.plot_temperature(temperature)
# visualization.plot_com_velocity(com_velocity, temperature, temperature_label=True)
# visualization.plot_com_ang_velocity(com_ang_velocity, temperature, temperature_label=True)

In [ ]:
area=40

visualization.plot_chiral_row_order(positions, handedness, area,
    spacing_range=(0.9, 1.6), frame_skip=100, temperature_label=True)

---

## Part 3: Visualize System's Configuration
* Plot staics and dynamics configuration of system's particles based on the selected target temperature, and for the whole of process:


In [ ]:
area, size = (40, 28)
area, size = (20, 90)

visualization.plot_snapshot_temperature(positions, handedness, alignment, 
                                        target_temperature=0.7, cooling=True,
                                        patchNums=1, 
                                        particle_size=size,
                                        line_length=.4, area=area, show_center=0,
                                        radius=True, 
                                        # save_name=interaction
                                        )

---

In [ ]:
visualization.plot_hist_phi_temperature(positions, 0.3, cooling=True, step_window=1000)

In [ ]:
visualization.plot_delta_phi_hist_temperature(positions, 0.1, cooling=True, min_dis=10, step_window=100)

---
* Create **animation** of particle's movement over simulation evolution:

In [ ]:
visualization.animate_position_handedness(positions, handedness, 'OP_heating.mp4', 
                                line_length=.4, area=area, 
                                color_p=False, 
                                frame_skip=10, frame_size=800, fps=20)

---

In [ ]:
import numpy as np
import pandas as pd
import adios2


def dat_to_bp(input_file, output_file, L0=7.0):

    # --------------------------------------------------------
    # Read box
    # --------------------------------------------------------
    box = np.loadtxt(input_file, max_rows=1)
    box_x, box_y = box

    if not np.isclose(box_x, box_y):
        raise ValueError("MD currently expects a square box.")

    # --------------------------------------------------------
    # Read MC particles
    # --------------------------------------------------------
    data = pd.read_csv(
        input_file,
        sep=r"\s+",
        header=None,
        skiprows=1,
        names=["x", "y", "phi", "zeta", "h", "d", "null"]
    )

    # --------------------------------------------------------
    # Same screw-period wrapping as old MC analysis
    # --------------------------------------------------------
    LE = data["zeta"].max()

    mask = data["zeta"] >= LE / 2
    data.loc[mask, "zeta"] -= LE
    data.loc[mask, "phi"] -= 100.0

    # --------------------------------------------------------
    # Convert to MD variables
    # --------------------------------------------------------
    areaL = box_x / L0

    x = (data["x"].to_numpy() / L0) % areaL
    y = (data["y"].to_numpy() / L0) % areaL

    positions = np.column_stack([
        x,
        y,
        np.deg2rad(data["phi"].to_numpy())
    ]).astype(np.float64)

    # vx, vy, omega = 0
    velocities = np.zeros_like(positions, dtype=np.float64)

    handedness = data["h"].to_numpy(dtype=np.int8)
    alignment  = data["d"].to_numpy(dtype=np.int8)

    N = len(data)
    D = 3

    print(f"N                 = {len(data)}")
    print(f"MC box            = {box_x:.6f} Å")
    print(f"Reduced box       = {areaL:.6f}")
    print(f"Reduced density   = {len(data) / areaL**2:.6f}")
    print(f"x range           = {x.min():.6f} ... {x.max():.6f}")
    print(f"y range           = {y.min():.6f} ... {y.max():.6f}")
    print("-"*50)
    xy = positions[:, :2]

    nearest = []

    for i in range(len(xy)):
        dr = xy - xy[i]
        dr -= areaL * np.round(dr / areaL)
        r = np.sqrt(np.sum(dr**2, axis=1))
        r[i] = np.inf
        nearest.append(r.min())

    nearest = np.asarray(nearest)

    print(f"nearest distance min  = {nearest.min():.4f}")
    print(f"nearest distance mean = {nearest.mean():.4f}")
    print(f"nearest distance max  = {nearest.max():.4f}")
    print("-"*50)
    # --------------------------------------------------------
    # Write BP in the SAME layout as C++
    # --------------------------------------------------------
    adios = adios2.Adios()
    io = adios.declare_io("SimulationOutput")

    time = np.array(0.0, dtype=np.float64)
    var_time = io.define_variable("time", time)
    var_positions = io.define_variable(
        "positions",
        positions,
        [N, D],
        [0, 0],
        [N, D]
    )

    var_velocities = io.define_variable(
        "velocities",
        velocities,
        [N, D],
        [0, 0],
        [N, D]
    )

    var_handedness = io.define_variable(
        "handedness",
        handedness,
        [N],
        [0],
        [N]
    )

    var_alignment = io.define_variable(
        "alignment",
        alignment,
        [N],
        [0],
        [N]
    )

    engine = io.open(output_file, adios2.Mode.Write)

    engine.begin_step()

    engine.put(var_time, time)
    engine.put(var_positions, positions)
    engine.put(var_velocities, velocities)
    engine.put(var_handedness, handedness)
    engine.put(var_alignment, alignment)

    engine.end_step()
    engine.close()

    print(f"Written: {output_file}")
    print(f"N = {N}")
    print(f"positions shape  = {positions.shape}")
    print(f"velocities shape = {velocities.shape}")
    print(f"box = {box_x / L0:.6f} reduced units")

In [ ]:
# target = 'EP/bcluster_65'
# target = 'EA/bcluster_89'
target = 'OP/bcluster_8'
# target = 'OA/bcluster_10-x'
# target = 'OAcorr/bcluster_3-x'


dat_to_bp(
    f"/home/hadis/polyalanine_paper/scripts/best_configs/{target}.dat",
    f"best_configs/{target}.bp",
    L0=7.0
)

In [ ]:
from adios2 import Stream

with Stream(f"best_configs/{target}.bp", "r") as bp:
    for _ in bp.steps():
        pos = bp.read("positions")
        h   = bp.read("handedness")
        d   = bp.read("alignment")

print(pos.shape)
print(pos[:5])
print(h[:5])
print(d[:5])